# BIRD Baseline — Qwen2.5-Coder-32B only (no schema linking)

Pure-LLM baseline for the **Stop-Guessing-Joins** ablation study. Same BIRD dev set, same model (`Qwen/Qwen2.5-Coder-32B-Instruct`, 4-bit NF4), but **no retriever, no graph resolver, no rewriter, no validator** — the model gets the *full* schema of each database plus the BIRD question/evidence and emits SQL directly.

This is ablation #2 from the project notes — the comparison point against the deterministic schema-linking pipeline in `bird_eval_eval.ipynb`. Predictions are written to `predict_dev_baseline.json` so they can be diffed against the pipeline run.

Designed for **Google Colab**. `HF_TOKEN` is read from Colab Secrets if present (Qwen is open-weights, so it's optional).

## 1. Setup — clone repo, install deps

Lighter dep set than the full-pipeline notebook: no `rank_bm25`, no `sentence-transformers`, no DFC rewriter — just `transformers`/`accelerate`/`bitsandbytes` for 4-bit inference and `sqlglot` for join-F1 metrics.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/mayursk2000/Stop-Guessing-Joins-Deterministic-Schema-Linking-for-Text-to-SQL-Agents.git"
REPO_DIR = pathlib.Path("/content/Stop-Guessing-Joins-Deterministic-Schema-Linking-for-Text-to-SQL-Agents")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "sqlglot", "huggingface_hub", "requests", "tqdm", "psutil",
    "transformers>=4.45", "accelerate>=0.34", "bitsandbytes>=0.43", "sentencepiece",
])
print("Repo:", REPO_DIR)

## 2. Secrets — `HF_TOKEN` from Colab Secrets (optional for Qwen)

In [ ]:
HF_TOKEN = None
try:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN not available:", e)
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print("HF_TOKEN set:", bool(HF_TOKEN))

## 3. GPU check — Qwen2.5-Coder-32B-Instruct (4-bit)

Baseline pins the 32B model regardless of VRAM so the comparison against the full-pipeline run is apples-to-apples. With 4-bit NF4 it fits in ~18 GB; the cell will warn if VRAM is below that.

In [ ]:
import torch

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  |  VRAM: {vram_gb:.1f} GB")
    if vram_gb < 18:
        print(f"WARNING: <18 GB VRAM. Qwen-32B in 4-bit needs ~18 GB. "
              f"Lower-VRAM options: Qwen2.5-Coder-14B-Instruct (~9 GB) or 7B-Instruct (~5 GB).")
else:
    print("No GPU detected. Set Runtime → Change runtime type → GPU.")

print("Locked model:", MODEL_NAME)

## 4. Download BIRD dev

Same downloader as the pipeline notebook — uses the public Aliyun mirror, falls back to gdown if a Google-Drive-style link is given.

In [ ]:
import re, zipfile, json, urllib.request

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
import gdown

BIRD_ROOT = pathlib.Path("/content/bird")
BIRD_ROOT.mkdir(exist_ok=True)

BIRD_DEV_URL = "https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip"
zip_path = BIRD_ROOT / "dev.zip"

def _looks_like_html(p: pathlib.Path) -> bool:
    if not p.exists() or p.stat().st_size < 1_000_000:
        try:
            head = p.read_bytes()[:512].lower()
            return b"<!doctype html" in head or b"<html" in head
        except Exception:
            return True
    return False

def _drive_id(url: str):
    for pat in (r"/file/d/([A-Za-z0-9_-]{20,})", r"[?&]id=([A-Za-z0-9_-]{20,})"):
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return None

if zip_path.exists() and _looks_like_html(zip_path):
    print(f"Existing {zip_path} looks like HTML ({zip_path.stat().st_size} bytes). Re-downloading.")
    zip_path.unlink()

if BIRD_DEV_URL and not zip_path.exists():
    drive_id = _drive_id(BIRD_DEV_URL)
    if drive_id:
        print(f"Google Drive download (id={drive_id}) ...")
        gdown.download(id=drive_id, output=str(zip_path), quiet=False)
    else:
        print("Direct download ...")
        urllib.request.urlretrieve(BIRD_DEV_URL, zip_path)
    print(f"Downloaded {zip_path.stat().st_size:,} bytes -> {zip_path}")

assert zip_path.exists(), "BIRD_DEV_URL is empty or download failed."
assert not _looks_like_html(zip_path), (
    f"{zip_path} is HTML ({zip_path.stat().st_size:,} bytes), not a zip."
)

if not list(BIRD_ROOT.rglob("dev.json")):
    print("Extracting ...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(BIRD_ROOT)
    for inner in BIRD_ROOT.rglob("dev_databases.zip"):
        with zipfile.ZipFile(inner) as z:
            z.extractall(inner.parent)

candidates = [p.parent for p in BIRD_ROOT.rglob("dev.json")]
assert candidates, f"dev.json not found under {BIRD_ROOT} after extraction."
BIRD_DEV = candidates[0]
BIRD_DBS = next(BIRD_DEV.rglob("dev_databases"), BIRD_DEV / "dev_databases")
print("BIRD_DEV =", BIRD_DEV)
print("BIRD_DBS =", BIRD_DBS)
print("Questions:", len(json.loads((BIRD_DEV / "dev.json").read_text())))
print("Databases:", len(list(BIRD_DBS.iterdir())) if BIRD_DBS.exists() else "missing")

## 5. Full-schema prompt builder

Baseline-specific helper. For each `db_id` it dumps **every** table with **every** column — no retrieval, no graph pruning. Column descriptions and FK metadata from BIRD's shipped files are appended for context (every BIRD submission uses these — fair game). The schema string is cached per db so we only build it once.

In [ ]:
import sqlite3, csv, json

_BIRD_TABLES_JSON: "dict | None" = None

def _load_dev_tables() -> dict:
    global _BIRD_TABLES_JSON
    if _BIRD_TABLES_JSON is None:
        path = next(BIRD_DEV.rglob("dev_tables.json"), None)
        if path is None:
            print("WARN: dev_tables.json not found — FK list will be empty.")
            _BIRD_TABLES_JSON = {}
        else:
            blocks = json.loads(path.read_text())
            _BIRD_TABLES_JSON = {b["db_id"]: b for b in blocks}
    return _BIRD_TABLES_JSON

def _read_column_descriptions(db_id: str) -> "dict[str, dict[str, str]]":
    desc_dir = BIRD_DBS / db_id / "database_description"
    out: "dict[str, dict[str, str]]" = {}
    if not desc_dir.exists():
        return out
    for csv_path in desc_dir.glob("*.csv"):
        tbl = csv_path.stem.lower()
        col_descs: "dict[str, str]" = {}
        try:
            with open(csv_path, encoding="utf-8", errors="replace") as f:
                for row in csv.DictReader(f):
                    col = (row.get("original_column_name") or "").strip()
                    desc = (row.get("column_description") or "").strip()
                    val_desc = (row.get("value_description") or "").strip()
                    full = " ; ".join(filter(None, [desc, val_desc]))
                    if col:
                        col_descs[col.lower()] = full
        except Exception as exc:
            print(f"  warn: couldn't read {csv_path.name}: {exc}")
        out[tbl] = col_descs
    return out

# Per-column description trim — keep prompts manageable on wide BIRD tables.
MAX_DESC_LEN = 100

def _trim(s: str) -> str:
    s = (s or "").replace("\r", " ").replace("\n", " ").replace("\t", " ").strip()
    if len(s) > MAX_DESC_LEN:
        s = s[:MAX_DESC_LEN].rstrip() + "…"
    return s

_SCHEMA_CACHE: "dict[str, str]" = {}

def full_schema_prompt(db_id: str) -> str:
    """Return a CREATE-TABLE-style dump of EVERY table/column in db_id, with
    BIRD column descriptions inlined and BIRD foreign keys appended at the end.
    No retrieval, no pruning — this is the baseline."""
    if db_id in _SCHEMA_CACHE:
        return _SCHEMA_CACHE[db_id]

    db_path = BIRD_DBS / db_id / f"{db_id}.sqlite"
    assert db_path.exists(), f"missing {db_path}"
    conn = sqlite3.connect(str(db_path))
    try:
        cur = conn.cursor()
        table_names = [r[0] for r in cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
        ).fetchall()]
        col_descs = _read_column_descriptions(db_id)

        blocks: "list[str]" = []
        for tname in table_names:
            cols_info = cur.execute(f'PRAGMA table_info("{tname}")').fetchall()
            tbl_descs = col_descs.get(tname.lower(), {})
            lines = []
            for c in cols_info:
                cname, ctype = c[1], (c[2] or "")
                desc = _trim(tbl_descs.get(cname.lower(), ""))
                base = f'    "{cname}" {ctype}'.rstrip()
                lines.append(f"{base}  -- {desc}" if desc else base)
            blocks.append(f'CREATE TABLE "{tname}" (\n' + ",\n".join(lines) + "\n);")

        # Foreign keys from BIRD's canonical metadata (preferred) + PRAGMA fallback.
        fk_lines: "list[str]" = []
        bird_meta = _load_dev_tables().get(db_id)
        seen = set()
        if bird_meta:
            names_orig = bird_meta["table_names_original"]
            cols_orig  = bird_meta["column_names_original"]
            for fk_pair in bird_meta.get("foreign_keys", []):
                l_idx, r_idx = fk_pair[0], fk_pair[1]
                lt = names_orig[cols_orig[l_idx][0]]; lc = cols_orig[l_idx][1]
                rt = names_orig[cols_orig[r_idx][0]]; rc = cols_orig[r_idx][1]
                key = (lt, lc, rt, rc)
                if key not in seen:
                    seen.add(key)
                    fk_lines.append(f'  "{lt}"."{lc}" -> "{rt}"."{rc}"')
        for tname in table_names:
            for fk in cur.execute(f'PRAGMA foreign_key_list("{tname}")').fetchall():
                _, _, ref_table, from_col, to_col, *_ = fk
                key = (tname, from_col, ref_table, to_col)
                if key not in seen:
                    seen.add(key)
                    fk_lines.append(f'  "{tname}"."{from_col}" -> "{ref_table}"."{to_col}"')

        schema_str = "\n\n".join(blocks)
        if fk_lines:
            schema_str += "\n\n-- Foreign keys:\n" + "\n".join(fk_lines)
        _SCHEMA_CACHE[db_id] = schema_str
        return schema_str
    finally:
        conn.close()

def load_dev(bird_dev_dir: pathlib.Path) -> list:
    return json.loads((bird_dev_dir / "dev.json").read_text())

## 6. Sanity check

In [ ]:
print("BIRD_DEV =", BIRD_DEV)
print("BIRD_DBS =", BIRD_DBS)
print(f"Found {len(list(BIRD_DBS.iterdir()))} databases")
questions = load_dev(BIRD_DEV)
print(f"Total questions: {len(questions)}")
print(f"\nSample schema dump for db_id={questions[0]['db_id']!r}:\n")
sample = full_schema_prompt(questions[0]["db_id"])
print(sample[:1500] + ("\n...[truncated]" if len(sample) > 1500 else ""))

## 7. Load Qwen2.5-Coder-32B (4-bit NF4) and define the baseline generator

Loads the model **once** and reuses it across all 1534 questions. The generator is intentionally minimal: chat-template prompt with system rules + full schema + evidence + question → first SELECT/WITH statement parsed out of the model output. No fallback to a deterministic stub — a malformed output is recorded as `SELECT 1` so the BIRD evaluator can still process it.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import re, gc

USE_4BIT          = True
MAX_NEW_TOKENS    = 256
MAX_INPUT_TOKENS  = 8192   # full-schema prompts are larger than linked-schema prompts (european_football_2 needs ~4.4k)

print(f"Loading {MODEL_NAME} (4-bit={USE_4BIT}) ...")

_bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"  # truncate schema from the front, never the question/instructions
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    quantization_config=_bnb_cfg,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Loaded. Device map:",
      model.hf_device_map if hasattr(model, "hf_device_map") else "single device")

_FENCE = re.compile(r"^```(?:sql|sqlite)?\s*|\s*```$", re.IGNORECASE | re.MULTILINE)

SYSTEM_PROMPT = (
    "You are an expert Text-to-SQL assistant for SQLite, evaluated on BIRD. "
    "You receive: (1) the FULL schema of one SQLite database, (2) optional "
    "BIRD evidence with column hints, (3) a natural-language question.\n\n"
    "STRICT RULES:\n"
    "- If the evidence names a specific column in backticks (e.g. "
    "`Free Meal Count (K-12)`), USE THAT EXACT COLUMN. Do not substitute a "
    "similar-sounding column even if it seems to mean the same thing.\n"
    "- If the evidence provides a formula (e.g. \"rate = A / B\"), COMPUTE "
    "that expression literally. When the formula is a ratio, CAST the numerator "
    "to REAL (e.g. CAST(A AS REAL) / B) so SQLite doesn't integer-divide.\n"
    "- If the evidence gives a literal value (e.g. \"= 1\" or \"= 'Directly funded'\"), "
    "use that EXACT value, including punctuation, capitalization, and spacing.\n"
    "- Use ONLY tables and columns that appear in the schema.\n"
    "- Quote column names containing spaces or special characters with double quotes.\n"
    "- Return EXACTLY ONE SQLite SELECT statement. No prose, no code fences."
)

def _build_messages(db_id: str, question: str, evidence: str) -> list:
    user_parts = [f"Database schema:\n{full_schema_prompt(db_id)}"]
    if evidence:
        user_parts.append(
            "BIRD evidence — REQUIRED column hints. If a column or value here is "
            f"given in backticks, use it EXACTLY as written:\n{evidence}"
        )
    user_parts.append(f"Question:\n{question}")
    user_parts.append("Output only the SELECT statement.")
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "\n\n".join(user_parts)},
    ]

def _clean(text: str) -> str:
    text = _FENCE.sub("", text).strip()
    m = re.search(r"\b(SELECT|WITH)\b", text, re.IGNORECASE)
    if m:
        text = text[m.start():]
    return text.strip().rstrip(";").strip()

def _validate(sql: str) -> str:
    if not re.match(r"(?i)^\s*(select|with)\b", sql):
        raise ValueError(f"Non-SELECT output: {sql[:120]}")
    return sql

def _free_cache():
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

def generate_sql_one(db_id: str, question: str, evidence: str) -> str:
    msgs = _build_messages(db_id, question, evidence)
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[0, inputs.input_ids.shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    del out, inputs, new_tokens
    _free_cache()
    return _validate(_clean(raw))

def generate_sql_batch(items: list, max_input_length: int = MAX_INPUT_TOKENS) -> list:
    """items: list of (db_id, question, evidence). Returns list of SQL strings (or 'SELECT 1' on per-item failure)."""
    if not items:
        return []
    out = inputs = new_token_rows = None
    try:
        prompts = [
            tokenizer.apply_chat_template(_build_messages(db, q, ev), tokenize=False, add_generation_prompt=True)
            for (db, q, ev) in items
        ]
        prompt_lengths = [len(tokenizer(p, add_special_tokens=False).input_ids) for p in prompts]
        if prompt_lengths and max(prompt_lengths) > max_input_length:
            print(f"  truncating batch prompts: max_tokens={max(prompt_lengths)} cap={max_input_length}")
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                           max_length=max_input_length).to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        input_len = inputs.input_ids.shape[1]
        new_token_rows = out[:, input_len:]
        decoded = tokenizer.batch_decode(new_token_rows, skip_special_tokens=True)
    except BaseException as exc:
        if isinstance(exc, (KeyboardInterrupt, SystemExit)):
            raise
        print(f"  batch failed ({type(exc).__name__}: {str(exc)[:120]})  batch={len(items)}")
        del out, inputs
        _free_cache(); gc.collect()
        if len(items) > 1:
            mid = len(items) // 2
            return generate_sql_batch(items[:mid], max_input_length) + generate_sql_batch(items[mid:], max_input_length)
        try:
            return [generate_sql_one(*items[0])]
        except Exception as inner:
            print(f"  per-item fallback failed: {inner}")
            return ["SELECT 1"]
    finally:
        try:
            del out, inputs, new_token_rows
        except NameError:
            pass
        _free_cache()

    results = []
    for raw in decoded:
        try:
            results.append(_validate(_clean(raw)))
        except Exception:
            results.append("SELECT 1")
    return results

print("Baseline generator ready.")

## 7b. Recover european_football_2 (one-time fix for the truncation bug)

Run this **once** if you have a previous `predict_dev_baseline.json` where european_football_2 was generated under a 4096-token cap with right-truncation. Skip on a fresh run — there's nothing to clean.

In [ ]:
# One-time fix: european_football_2 schema (~13k chars / ~4.4k tokens) was being truncated
# from the right under the old 4096 cap, so the question got chopped and all 129 predictions
# collapsed to the same output. With MAX_INPUT_TOKENS=8192 + truncation_side="left" above, the
# fix is in. Wipe those predictions so cell 18's resume logic redoes them.
import json, pathlib
_PRED_PATH = pathlib.Path("/content/predict_dev_baseline.json")
if _PRED_PATH.exists():
    _preds = json.loads(_PRED_PATH.read_text())
    _victims = [qid for qid, line in _preds.items() if line.endswith("\teuropean_football_2")]
    if _victims:
        for qid in _victims:
            del _preds[qid]
        _PRED_PATH.write_text(json.dumps(_preds))
        print(f"Removed {len(_victims)} european_football_2 predictions; {len(_preds)} remain. "
              f"Re-run cell 18 to regenerate them with the fixed prompt.")
    else:
        print("No european_football_2 predictions to remove (already clean).")
else:
    print("No predictions file yet — nothing to clean.")

## 8. Smoke test — one BIRD question end-to-end

Sanity check before burning a full run.

In [ ]:
q0 = questions[0]
print("db_id:    ", q0["db_id"])
print("question: ", q0["question"])
print("evidence: ", q0.get("evidence", ""))
print("gold SQL: ", q0["SQL"])
print("-" * 80)

sql = generate_sql_one(q0["db_id"], q0["question"], q0.get("evidence", "") or "")
print("\nBaseline SQL:", sql)

## 9. Run on full BIRD dev → `predict_dev_baseline.json`

Predictions are grouped by `db_id` so the schema cache for each db is hit warm, processed in batches of `BASELINE_BATCH_SIZE`, checkpointed every 50 questions. Output format matches BIRD's official evaluator (`<SQL>\t----- bird -----\t<db_id>`).

In [ ]:
from tqdm.auto import tqdm
from collections import defaultdict
import gc, psutil, os, torch, json

N             = len(questions)
BATCH_SIZE    = int(os.environ.get("BASELINE_BATCH_SIZE", "8"))
PRED_PATH     = pathlib.Path("/content/predict_dev_baseline.json")

predictions: "dict[str, str]" = {}
if PRED_PATH.exists():
    predictions = json.loads(PRED_PATH.read_text())
    print(f"Resuming with {len(predictions)} predictions already on disk")

def _mem_str() -> str:
    rss = psutil.Process(os.getpid()).memory_info().rss / 1e9
    avail = psutil.virtual_memory().available / 1e9
    gpu = ""
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        gpu = f"  GPU free={free/1e9:.1f}GB"
    return f"RSS={rss:.1f}GB  free={avail:.1f}GB{gpu}"

def write_predictions(path: pathlib.Path, preds: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(preds, f)

groups: "dict[str, list]" = defaultdict(list)
for i, q in enumerate(questions[:N]):
    groups[q["db_id"]].append((i, q))

print(f"BIRD dev: {N} questions across {len(groups)} databases")
print(f"Memory at start: {_mem_str()}")

total_done = len(predictions)
pbar = tqdm(total=N, initial=total_done, desc="BIRD dev (baseline)")
last_checkpoint = total_done

for db_id, qlist in groups.items():
    qlist_remaining = [(i, q) for (i, q) in qlist if str(q.get("question_id", i)) not in predictions]
    if not qlist_remaining:
        continue

    # Warm the schema cache for this db (also lets us print its size).
    sch = full_schema_prompt(db_id)
    print(f"  [{db_id}] {len(qlist_remaining)} questions, schema_chars={len(sch):,}  {_mem_str()}")

    # Process this db in batches.
    pending: "list[tuple[str, dict]]" = []  # (qid, q)
    def flush():
        global last_checkpoint
        if not pending:
            return
        items = [(db_id, q["question"], q.get("evidence", "") or "") for (_qid, q) in pending]
        sqls = generate_sql_batch(items)
        for (qid, _q), sql in zip(pending, sqls):
            sql = (sql or "SELECT 1").replace("\n", " ").strip()
            predictions[qid] = f"{sql}\t----- bird -----\t{db_id}"
            pbar.update(1)
        pending.clear()
        gc.collect()
        try: torch.cuda.empty_cache()
        except Exception: pass
        if len(predictions) - last_checkpoint >= 50:
            write_predictions(PRED_PATH, predictions)
            last_checkpoint = len(predictions)

    for orig_i, q in qlist_remaining:
        qid = str(q.get("question_id", orig_i))
        if qid in predictions:
            continue
        pending.append((qid, q))
        if len(pending) >= BATCH_SIZE:
            flush()

    flush()
    write_predictions(PRED_PATH, predictions)
    print(f"  [{db_id}] done. {_mem_str()}")

pbar.close()
write_predictions(PRED_PATH, predictions)
print(f"\nWrote {PRED_PATH} ({len(predictions)} predictions)")
print(f"Memory at end: {_mem_str()}")

## 10. Local metrics — Join Accuracy + Execution Accuracy

Same EX/Join-F1 metrics the pipeline notebook reports, so the two runs are directly comparable. (Graph-table-recall is omitted — there's no graph in this baseline.) For leaderboard-style EX/VES use BIRD's official scripts in §11.

In [ ]:
import sqlite3, json, time
from collections import defaultdict
import sqlglot
from sqlglot import exp

PRED_PATH = pathlib.Path("/content/predict_dev_baseline.json")
predictions = json.loads(PRED_PATH.read_text())
qmap = {str(q.get("question_id", i)): q for i, q in enumerate(questions)}

TIMEOUT_S = 10
SQLITE_PROGRESS_STEPS = 10_000
_NULL = "\x00NULL\x00"

def run_sql(db_id: str, sql: str):
    db_path = BIRD_DBS / db_id / f"{db_id}.sqlite"
    conn = sqlite3.connect(str(db_path), timeout=TIMEOUT_S)
    try:
        conn.execute(f"PRAGMA busy_timeout = {TIMEOUT_S * 1000}")
        deadline = time.monotonic() + TIMEOUT_S
        conn.set_progress_handler(lambda: 1 if time.monotonic() > deadline else 0, SQLITE_PROGRESS_STEPS)
        return conn.execute(sql).fetchall(), None
    except Exception as exc:
        return None, str(exc)
    finally:
        try:
            conn.set_progress_handler(None, 0)
        except Exception:
            pass
        conn.close()

def normalize(rows):
    if rows is None:
        return None
    return sorted(tuple(_NULL if c is None else str(c) for c in r) for r in rows)

def extract_joins(sql: str):
    try:
        tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception:
        return set()
    alias_map: "dict[str, str]" = {}
    for t in tree.find_all(exp.Table):
        name = (t.name or "").lower()
        alias = (t.alias or t.name or "").lower()
        if name and alias:
            alias_map[alias] = name
    joins = set()
    for eq in tree.find_all(exp.EQ):
        left, right = eq.this, eq.expression
        if isinstance(left, exp.Column) and isinstance(right, exp.Column):
            lt = (left.table or "").lower()
            rt = (right.table or "").lower()
            if lt and rt and lt in alias_map and rt in alias_map and lt != rt:
                a = f"{alias_map[lt]}.{left.name.lower()}"
                b = f"{alias_map[rt]}.{right.name.lower()}"
                joins.add(frozenset([a, b]))
    return joins

def join_f1(pred_sql: str, gold_sql: str):
    gold = extract_joins(gold_sql)
    if not gold:
        return None
    pred = extract_joins(pred_sql)
    if not pred:
        return 0.0
    tp = len(pred & gold)
    if tp == 0:
        return 0.0
    p = tp / len(pred); r = tp / len(gold)
    return 2 * p * r / (p + r)

results = []
for i, q in enumerate(questions):
    qid = str(q.get("question_id", i))
    line = predictions.get(qid)
    if line is None:
        pred_sql = "SELECT 1"; missing = True
    else:
        pred_sql = line.split("\t----- bird -----\t", 1)[0].strip(); missing = False
    is_fallback = pred_sql.rstrip(";").strip().upper() == "SELECT 1"

    pred_rows, pred_err = run_sql(q["db_id"], pred_sql)
    gold_rows, gold_err = run_sql(q["db_id"], q["SQL"])

    if missing: ex = "missing_prediction"
    elif pred_err:      ex = "exec_error"
    elif gold_err:      ex = "gold_error"
    elif normalize(pred_rows) == normalize(gold_rows): ex = "correct"
    else:               ex = "wrong_result"

    results.append({
        "qid": qid, "db_id": q["db_id"], "difficulty": q.get("difficulty", "?"),
        "ex": ex, "pred_err": pred_err, "fallback": is_fallback, "missing": missing,
        "jf1": join_f1(pred_sql, q["SQL"]), "pred_sql": pred_sql,
    })

total = len(results)
ex_correct = sum(r["ex"] == "correct" for r in results)
ex_acc = ex_correct / max(total, 1) * 100
scored_join = [r for r in results if r["jf1"] is not None]
join_acc = (sum(r["jf1"] for r in scored_join) / max(len(scored_join), 1)) * 100

print(f"=== Baseline (Qwen2.5-Coder-32B, full schema, no pipeline) — {total} predictions ===\n")
print(f"{'Method':<28s}{'Join Acc.':>12s}{'Exec. Acc.':>13s}")
print(f"{'-'*53}")
print(f"{'Prior Work [BIRD]':<28s}{58.2:>12.1f}{54.9:>13.1f}")
print(f"{'Agent-based':<28s}{63.7:>12.1f}{57.1:>13.1f}")
print(f"{'Ours (paper target)':<28s}{73.8:>12.1f}{64.4:>13.1f}")
print(f"{'This run (baseline)':<28s}{join_acc:>12.1f}{ex_acc:>13.1f}")

print(f"\nJoin Acc. averaged over {len(scored_join)} multi-table questions; "
      f"{total - len(scored_join)} single-table excluded.")
print(f"\nEX status:  correct={ex_correct}  "
      f"wrong={sum(r['ex']=='wrong_result' for r in results)}  "
      f"err={sum(r['ex']=='exec_error' for r in results)}  "
      f"missing={sum(r['ex']=='missing_prediction' for r in results)}  "
      f"fallback={sum(r['fallback'] for r in results)}")

print(f"\n--- By difficulty ---")
print(f"{'difficulty':<14s}{'Join Acc.':>11s}{'Exec. Acc.':>12s}{'cases':>8s}")
for d in ("simple", "moderate", "challenging", "?"):
    bucket = [r for r in results if r["difficulty"] == d]
    if not bucket:
        continue
    bj = [r for r in bucket if r["jf1"] is not None]
    j = (sum(r["jf1"] for r in bj) / max(len(bj), 1)) * 100 if bj else float("nan")
    e = sum(r["ex"] == "correct" for r in bucket) / len(bucket) * 100
    print(f"  {d:<12s}{j:>11.1f}{e:>12.1f}{len(bucket):>8d}")

print(f"\n--- By db_id ---")
print(f"{'db_id':<26s}{'Join Acc.':>11s}{'Exec. Acc.':>12s}{'cases':>8s}")
by_db: "dict[str, list]" = defaultdict(list)
for r in results:
    by_db[r["db_id"]].append(r)
for d, bucket in sorted(by_db.items()):
    bj = [r for r in bucket if r["jf1"] is not None]
    j = (sum(r["jf1"] for r in bj) / max(len(bj), 1)) * 100 if bj else float("nan")
    e = sum(r["ex"] == "correct" for r in bucket) / len(bucket) * 100
    print(f"  {d:<24s}{j:>11.1f}{e:>12.1f}{len(bucket):>8d}")

print("\n--- First 5 EX failures ---")
shown = 0
for r in results:
    if r["ex"] == "correct" or shown >= 5:
        continue
    q = qmap[r["qid"]]
    jf = "—" if r["jf1"] is None else f"{r['jf1']:.2f}"
    print(f"\n  qid={r['qid']} ({r['difficulty']}) db={r['db_id']}  ex={r['ex']}  join_f1={jf}")
    print(f"    Q:    {q['question'][:140]}")
    if q.get("evidence"):
        print(f"    ev:   {q['evidence'][:140]}")
    print(f"    pred: {r['pred_sql'][:160]}")
    print(f"    gold: {q['SQL'][:160]}")
    if r["pred_err"]:
        print(f"    err:  {r['pred_err'][:160]}")
    shown += 1

## 11. Run BIRD's official evaluation scripts

Drop `evaluation.py` and `evaluation_ves.py` from BIRD's `evaluation/` folder into `/content/bird_eval/`. The cell builds the gold-SQL file in BIRD's expected format and invokes both scripts on the baseline predictions.

In [ ]:
BIRD_EVAL = pathlib.Path("/content/bird_eval")
BIRD_EVAL.mkdir(exist_ok=True)

GT_PATH = BIRD_EVAL / "dev_gold.sql"
lines = []
for q in questions:
    sql = q["SQL"].replace("\n", " ").strip()
    lines.append(f"{sql}\t{q['db_id']}")
GT_PATH.write_text("\n".join(lines))
print("Wrote", GT_PATH)

ex_script  = BIRD_EVAL / "evaluation.py"
ves_script = BIRD_EVAL / "evaluation_ves.py"

if ex_script.exists():
    !python {ex_script} \
        --predicted_sql_path {PRED_PATH} \
        --ground_truth_path {GT_PATH} \
        --db_root_path {BIRD_DBS} \
        --num_cpus 4 \
        --meta_time_out 30.0 \
        --mode_gt gt --mode_predict gpt
else:
    print(f"Place evaluation.py at {ex_script} and re-run.")

if ves_script.exists():
    !python {ves_script} \
        --predicted_sql_path {PRED_PATH} \
        --ground_truth_path {GT_PATH} \
        --db_root_path {BIRD_DBS} \
        --num_cpus 4 \
        --meta_time_out 30.0 \
        --mode_gt gt --mode_predict gpt
else:
    print(f"Place evaluation_ves.py at {ves_script} and re-run.")

## Notes

**This is the baseline.** Same model, same data, no schema linking. The delta in **Exec. Acc.** between this run and `bird_eval_eval.ipynb` is the contribution attributable to the deterministic schema-linking pipeline (retriever + graph + rewriter).

**Throughput.** With 4-bit Qwen2.5-Coder-32B on an A100 40GB, expect ~3–5 sec/question greedy → ~1.5–2.5 hours for full dev (1534 Q). Slightly slower than the pipeline notebook because full-schema prompts are longer than linked-schema prompts. For production-scale eval swap `transformers.generate` for **vLLM** (5–10× speedup).

**Predictions file.** `/content/predict_dev_baseline.json` (separate from the pipeline run's `predict_dev.json`) so both can coexist for direct comparison.